# Compute or update `shuga` metrics

Notebook-cell version of the `metrics.py` script.

This notebook replaces command-line arguments with editable variables and keeps the same high-level workflow:

1. configure the run, classification, metric, observation, plotting, and path settings;
2. construct the `shuga` runtime objects;
3. resolve/log the relevant stores;
4. compute or update metrics for one or more classification methods;
5. optionally write common FIP/FIA/FIT/SIA/SIT plots.


## Imports and repository path


In [1]:
import sys
from pathlib import Path
#####################################################################
# make sure this reflects the correct location of mawsons-chest repo
repo_root = Path.home() / "AFIM" / "src" / "mawsons-chest"
#####################################################################
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from shuga              import (CICEMetrics,
                                CICEPlotter,
                                ClassificationSpec,
                                MetricsSpec,
                                ObservationSpec,
                                PlottingSpec,
                                RunSpec,
                                ShugaPaths)
from shuga.core.naming  import normalize_method
from shuga.core.logging import build_file_logger
def comma_split(value: str | None) -> list[str]:
    if value is None:
        return []
    return [v.strip() for v in value.split(",") if v.strip()]

## run context


In [2]:
SIM_NAME       = "LD-fsnow-sep"
START_DATE     = "1994-01-01"
END_DATE       = "1994-12-31"
HEMISPHERE     = "SH"
PROJECT        = "gv90"
USER           = "da1339"
ICEH_FREQUENCY = "daily"       # "daily" or "hourly"

## classification context


In [3]:
ICE_TYPE      = "FI"
GRID_TYPE     = "Tc"
ISPD_THRESH   = 5.0e-4
METHODS       = "binary-days,rolling-mean"
BIN_WINDOW    = 11
BIN_MIN_DAYS  = 9
ROLL_WINDOW   = 15

## metrics configuration


In [20]:
METRIC_GROUPS             = "fi_core,pi_core,si_core"
METRIC_NAMES              = None
UPDATE_MISSING_ONLY       = True
OVERWRITE                 = False
REBUILD_ON_INDEX_MISMATCH = False
OBS_METRICS_STORE         = None
OBS_FIA_VAR               = "FIA"
OBS_FIT_VAR               = "FIT"
COAST_DISTANCE_VAR        = None

## observations, paths, and plotting config


In [21]:
CICE_STORE           = None
STATIC_STORE         = None
CLASSIFICATION_ROOT  = None
AFIM_OUTPUT_ROOT     = None
GRAPHICS_ROOT        = None
LOGS_ROOT            = None
SEAICE_ROOT          = None
NSIDC_ROOT           = None
NSIDC_CELLAREA_ROOT  = None
AF2020_ROOT          = None
PLOT_FIP             = True
PLOT_FIA             = True
PLOT_FIT             = True
PLOT_SIA             = True
PLOT_SIT             = True
PLOT_REGION          = "total"
FIP_REGION           = None
FIG_SIZE             = 20.0
LOG_LEVEL            = "INFO"

## Build method and metric request lists


In [22]:
methods       = [normalize_method(m) for m in comma_split(METHODS)]
metric_groups = comma_split(METRIC_GROUPS)
metric_names  = comma_split(METRIC_NAMES)
print(f"methods       : {methods}")
print(f"metric_groups : {metric_groups}")
print(f"metric_names  : {metric_names}")


methods       : ['binary-days', 'rolling-mean']
metric_groups : ['fi_core', 'pi_core', 'si_core']
metric_names  : []


## Build `shuga` runtime objects


In [23]:
run_cfg       = RunSpec(sim_name       = SIM_NAME,
                        start_date     = START_DATE,
                        end_date       = END_DATE,
                        hemisphere     = HEMISPHERE,
                        project        = PROJECT,
                        user           = USER,
                        iceh_frequency = ICEH_FREQUENCY)
cls_cfg       = ClassificationSpec(ice_type     = ICE_TYPE,
                                   grid_type    = GRID_TYPE,
                                   ispd_thresh  = ISPD_THRESH,
                                   methods      = tuple(methods),
                                   bin_window   = BIN_WINDOW,
                                   bin_min_days = BIN_MIN_DAYS,
                                   roll_window  = ROLL_WINDOW)
met_cfg       = MetricsSpec(methods            = tuple(methods),
                            obs_metrics_store  = OBS_METRICS_STORE,
                            obs_fia_var        = OBS_FIA_VAR,
                            obs_fit_var        = OBS_FIT_VAR,
                            coast_distance_var = COAST_DISTANCE_VAR)
obs_cfg       = ObservationSpec(seaice_root         = SEAICE_ROOT,
                                nsidc_root          = NSIDC_ROOT,
                                nsidc_cellarea_root = NSIDC_CELLAREA_ROOT,
                                af2020_root         = AF2020_ROOT)
plt_cfg       = PlottingSpec(fig_size        = FIG_SIZE,
                             fip_fig_size    = FIG_SIZE,
                             region_fig_size = FIG_SIZE)
pth_cfg       = ShugaPaths(run_cfg             = run_cfg,
                           cls_cfg             = cls_cfg,
                           obs_cfg             = obs_cfg,
                           afim_output_root    = AFIM_OUTPUT_ROOT,
                           graphics_root       = GRAPHICS_ROOT,
                           cice_store          = CICE_STORE,
                           static_store        = STATIC_STORE,
                           classification_root = CLASSIFICATION_ROOT,
                           logs_root           = LOGS_ROOT)
print(run_cfg)
print(cls_cfg)
print(met_cfg)

RunSpec(sim_name='LD-fsnow-sep', start_date='1994-01-01', end_date='1994-12-31', hemisphere='SH', project='gv90', user='da1339', iceh_frequency='daily')
ClassificationSpec(ice_type='FI', grid_type='Tc', ispd_thresh=0.0005, methods=('binary-days', 'rolling-mean'), bin_window=11, bin_min_days=9, roll_window=15, speed_var_u='uvel', speed_var_v='vvel', uvelE_var='uvelE', uvelN_var='uvelN', vvelE_var='vvelE', vvelN_var='vvelN', aice_var='aice', aice_thresh=0.15, wrap_x=True, cgrid_combine='mean')
MetricsSpec(methods=('binary-days', 'rolling-mean'), obs_metrics_store=None, obs_fia_var='FIA', obs_fit_var='FIT', coast_distance_var=None, area_scale=1000000000.0, volume_scale=1000000000000.0)


## Logging and path sanity checks


In [24]:
logger        = build_file_logger("shuga.met_cfg",
                                  pth_cfg.metrics_log_path(),
                                  level = LOG_LEVEL)
logger.info("Logging to: %s", pth_cfg.metrics_log_path())
logger.info("Resolved CICE store: %s", pth_cfg.resolve_cice_store())
print(f"Log file             : {pth_cfg.metrics_log_path()}")
print(f"Resolved CICE store  : {pth_cfg.resolve_cice_store()}")
static_store  = pth_cfg.resolve_static_store()
if static_store is None:
    logger.warning("No CICE static store resolved. Metrics requiring tarea/TLON/TLAT or face areas may fail.")
    print("WARNING: no CICE static store resolved")
else:
    logger.info("Resolved static store: %s", static_store)
    print(f"Resolved static store: {static_store}")
logger.info("Resolved classification root: %s", pth_cfg.classification_root_path)
print(f"Classification root  : {pth_cfg.classification_root_path}")

2026-06-07 14:11:12,974 - INFO - [shuga.met_cfg.<module>:4] Logging to: /g/data/gv90/da1339/logs/metrics/metrics_LD-fsnow-sep_FI_Tc_ispd_thresh5e-4_BW11_BM9_roll15.log
2026-06-07 14:11:12,976 - INFO - [shuga.met_cfg.<module>:5] Resolved CICE store: /g/data/gv90/da1339/afim_output/LD-fsnow-sep/zarr/iceh_daily.zarr
2026-06-07 14:11:12,980 - INFO - [shuga.met_cfg.<module>:13] Resolved static store: /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr
2026-06-07 14:11:12,981 - INFO - [shuga.met_cfg.<module>:15] Resolved classification root: /home/581/da1339/AFIM_archive/LD-fsnow-sep/zarr/SH/ispd_thresh_5.0e-4/FI/Tc


Log file             : /g/data/gv90/da1339/logs/metrics/metrics_LD-fsnow-sep_FI_Tc_ispd_thresh5e-4_BW11_BM9_roll15.log
Resolved CICE store  : /g/data/gv90/da1339/afim_output/LD-fsnow-sep/zarr/iceh_daily.zarr
Resolved static store: /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr
Classification root  : /home/581/da1339/AFIM_archive/LD-fsnow-sep/zarr/SH/ispd_thresh_5.0e-4/FI/Tc


## Instantiate metrics runner and plotter


In [25]:
runner              = CICEMetrics(run_cfg = run_cfg,
                                  cls_cfg = cls_cfg,
                                  met_cfg = met_cfg,
                                  pth_cfg = pth_cfg,
                                  logger  = logger)
plotter             = CICEPlotter(run_cfg = run_cfg,
                                  cls_cfg = cls_cfg,
                                  met_cfg = met_cfg,
                                  plt_cfg = plt_cfg,
                                  obs_cfg = obs_cfg,
                                  pth_cfg = pth_cfg,
                                  logger  = logger)
update_missing_only = UPDATE_MISSING_ONLY or (not OVERWRITE)
print(f"update_missing_only: {update_missing_only}")

update_missing_only: True


## Compute or update metrics


In [32]:
for method in methods:
    logger.info("Processing class method: %s", method)
    print(f"Processing class method: {method}")
    for met_grp in metric_groups:
        print(f"Processing metric group: {met_grp}")
        runner.compute_metrics(method,
                               overwrite                 = OVERWRITE,
                               metric_groups             = met_grp,
                               metric_names              = metric_names,
                               update_missing_only       = update_missing_only,
                               rebuild_on_index_mismatch = REBUILD_ON_INDEX_MISMATCH)


2026-06-07 14:22:31,359 - INFO - [shuga.met_cfg.<module>:2] Processing class method: binary-days
2026-06-07 14:22:31,363 - INFO - [shuga.met_cfg.compute_metrics:819] Resolved class store for binary-days: /home/581/da1339/AFIM_archive/LD-fsnow-sep/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/data.zarr
2026-06-07 14:22:31,365 - INFO - [shuga.met_cfg.compute_metrics:820] Resolved metrics store for binary-days: /home/581/da1339/AFIM_archive/LD-fsnow-sep/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
2026-06-07 14:22:31,367 - INFO - [shuga.met_cfg.compute_metrics:832] All requested metrics already present for binary-days; nothing to do.


Processing class method: binary-days
Processing metric group: fi_core
Processing metric group: pi_core


ValueError: cls_cfg.ice_type='FI' but requested metrics are for ['PI']. Use --ice-type PI.

## Optional plots


In [11]:
for method in methods:
    if PLOT_FIP:
        P_pngs = plotter.plot_fip(method, region_name = FIP_REGION if FIP_REGION not in {None, "", "total"} else None)
        logger.info("Wrote FIP plot(s): %s", P_pngs)
        print(f"[{method}]")
        for reg_name, P_png in P_pngs.items()
            print(f"FIP: {P_png}")
            display(Image(filename = P_png))
    if PLOT_FIA:
        P_png = plotter.plot_timeseries("FIA", method, region = PLOT_REGION)
        logger.info("Wrote FIA plot: %s", P_png)
        print(f"[{method}] FIA: {P_png}")
        display(Image(filename = P_png))
    if PLOT_FIT:
        P_png = plotter.plot_timeseries("FIT", method, region = PLOT_REGION, add_f2020 = False)
        logger.info("Wrote FIT plot: %s", P_png)
        print(f"[{method}] FIT: {P_png}")
        display(Image(filename = P_png))
    if PLOT_SIA:
        P_png = plotter.plot_timeseries("SIA", method, region = PLOT_REGION, add_f2020 = False)
        logger.info("Wrote SIA plot: %s", P_png)
        print(f"[{method}] SIA: {P_png}")
        display(Image(filename = P_png))
    if PLOT_SIT:
        P_png = plotter.plot_timeseries("SIT", method, region = PLOT_REGION, add_f2020 = False)
        logger.info("Wrote SIT plot: %s", P_png)
        print(f"[{method}] SIT: {P_png}")
        display(Image(filename = P_png))

2026-06-07 14:00:36,354 - INFO - [shuga.met_cfg._log:57] Opening CICE static-coordinate store: /home/581/da1339/AFIM_archive/CICE_0p25_Cgrid_coords.zarr
2026-06-07 14:00:57,120 - INFO - [shuga.met_cfg.<module>:4] Wrote FIP plot(s): {'DML': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/DML/LD-fsnow-sep_FIP_binary-days.png', 'WIO': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/WIO/LD-fsnow-sep_FIP_binary-days.png', 'EIO': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/EIO/LD-fsnow-sep_FIP_binary-days.png', 'Aus': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/Aus/LD-fsnow-sep_FIP_binary-days.png', 'VOL': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/VOL/LD-fsnow-sep_FIP_binary-days.png', 'AS': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/AS/LD-fsnow-sep_FIP_binary-days.png', 'BS': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/BS/LD-fsnow-sep_FIP_binary-days.png', 'WS': '/g/data/gv90/da

[binary-days] FIP: {'DML': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/DML/LD-fsnow-sep_FIP_binary-days.png', 'WIO': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/WIO/LD-fsnow-sep_FIP_binary-days.png', 'EIO': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/EIO/LD-fsnow-sep_FIP_binary-days.png', 'Aus': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/Aus/LD-fsnow-sep_FIP_binary-days.png', 'VOL': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/VOL/LD-fsnow-sep_FIP_binary-days.png', 'AS': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/AS/LD-fsnow-sep_FIP_binary-days.png', 'BS': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/BS/LD-fsnow-sep_FIP_binary-days.png', 'WS': '/g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/FIP/LD-fsnow-sep/WS/LD-fsnow-sep_FIP_binary-days.png'}


2026-06-07 14:01:03,790 - INFO - [shuga.met_cfg.plot_timeseries:826] creating figure
2026-06-07 14:01:03,814 - INFO - [shuga.met_cfg.plot_timeseries:842]      basemap
2026-06-07 14:01:03,850 - INFO - [shuga.met_cfg.plot_timeseries:845]      observations
2026-06-07 14:01:03,851 - INFO - [shuga.met_cfg.plot_timeseries:846]                   time       value
0  1994-01-01T00:00:00  386.304413
1  1994-01-02T00:00:00  382.626740
2  1994-01-03T00:00:00  378.777588
3  1994-01-04T00:00:00  374.786438
4  1994-01-05T00:00:00  370.682709
2026-06-07 14:01:03,893 - INFO - [shuga.met_cfg.plot_timeseries:848]      model results
2026-06-07 14:01:03,924 - INFO - [shuga.met_cfg.plot_timeseries:850]      saving
2026-06-07 14:01:04,249 - INFO - [shuga.met_cfg.<module>:8] Wrote FIA plot: /g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/total/timeseries/1994-01-01_1994-12-31_LD-fsnow-sep_FIA_binary_days.png
2026-06-07 14:01:04,251 - INFO - [shuga.met_cfg.plot_timeseries:750] loading metrics for LD-fsnow-sep over 

[binary-days] FIA: /g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/total/timeseries/1994-01-01_1994-12-31_LD-fsnow-sep_FIA_binary_days.png


2026-06-07 14:01:04,479 - INFO - [shuga.met_cfg.plot_timeseries:848]      model results
2026-06-07 14:01:04,510 - INFO - [shuga.met_cfg.plot_timeseries:850]      saving
2026-06-07 14:01:04,756 - INFO - [shuga.met_cfg.<module>:12] Wrote FIT plot: /g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/total/timeseries/1994-01-01_1994-12-31_LD-fsnow-sep_FIT_binary_days.png
2026-06-07 14:01:04,758 - INFO - [shuga.met_cfg.plot_timeseries:750] loading metrics for LD-fsnow-sep over period 1994-01-01 -- 1994-12-31 for hemisphere SH ...
INFO:shuga.io.zarr_loading:Resolving run context.
INFO:shuga.io.zarr_loading:Resolving classification context.
INFO:shuga.io.zarr_loading:Building ShugaPaths.
INFO:shuga.io.zarr_loading:Resolved metrics store for LD-fsnow-sep [Tc/binary-days] by exact path: /home/581/da1339/AFIM_archive/LD-fsnow-sep/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr
INFO:shuga.io.zarr_loading:Opening metrics store for LD-fsnow-sep [Tc/binary-days]: /home/581/da1339/AFIM_archiv

[binary-days] FIT: /g/data/gv90/da1339/GRAPHICAL/LD-fsnow-sep/total/timeseries/1994-01-01_1994-12-31_LD-fsnow-sep_FIT_binary_days.png


KeyError: "Variable 'SIA' not found in metrics dataset."

## Inspect the metrics stores


In [ ]:
from shuga import load_metrics

for method in methods:
    try:
        ds = load_metrics(run_cfg        = run_cfg,
                          cls_cfg        = cls_cfg,
                          met_cfg        = met_cfg,
                          pth_cfg        = pth_cfg,
                          classification = method,
                          variables      = None)
        print(f"\n[{method}]")
        print(ds)
    except Exception as exc:
        print(f"Could not open metrics output for {method}: {exc}")
